# Create a household population with a conditional WGAN

This workflow learns household relationships from the weighted seed and directly generates a population whose household-size counts exactly match every census zone. It does not create a pool. Census data is not a training prior: it supplies the household-size controls and the explicit Negative-income rate.

Because the seed has no `Negative income` records, weighted copies of `Nil income` households are relabelled within each `hhsize`. This assumes both income states have the same non-income distribution within household size.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch

from PopSynthesis.Methods.GAN.wgan import (
    GENERATED_ATTRIBUTES, HOUSEHOLD_ATTRIBUTES, WGANConfig, file_sha256,
    load_wgan_artifact, prepare_household_training_data,
    sample_zonal_exact_marginal, save_wgan_artifact,
    train_household_wgan,
)

## Configuration

In [2]:
REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists())
SEED_FILE = REPO_ROOT / 'IO/processed_data/ori_sample_hh.csv'
CENSUS_FILE = REPO_ROOT / 'IO/processed_data/census_POA_hh_saa.csv'
MODEL_FILE = REPO_ROOT / 'IO/models/hh_wgan/household_wgan.pt'
OUTPUT_FILE = REPO_ROOT / 'IO/output/hh_wgan_exact_hhsize.csv'

TRAIN_MODEL = True
NEGATIVE_INCOME_PROBABILITY = None  # None uses the raw census marginal
CHUNK_SIZE = 100_000
RANDOM_SEED = 42
CONFIG = WGANConfig(epochs=100, batch_size=512, hidden_dim=128, seed=RANDOM_SEED)

## Prepare the weighted training data

In [3]:
seed = pd.read_csv(SEED_FILE)
census = pd.read_csv(CENSUS_FILE, header=[0, 1])
prepared = prepare_household_training_data(
    seed=seed, census=census,
    negative_income_probability=NEGATIVE_INCOME_PROBABILITY,
)
print('Negative-income probability:', f'{prepared.negative_income_probability:.6%}')
display(pd.DataFrame(prepared.diagnostics['strata']))
display(pd.DataFrame(prepared.income_given_hhsize).T[['Negative income', 'Nil income']])

Negative-income probability: 0.707952%


,hhsize,original_weight,negative_weight,donor_rows
0,1,991288.77,7017.847455,816
1,2,1468136.65,10393.700972,748
2,3,658855.10,4664.377048,233
3,4,698403.18,4944.358423,223
4,5,240127.95,1699.990330,87
5,6,62083.37,439.520383,18
6,7,13853.99,98.079582,6
7,8+,6496.33,45.990890,1


,Negative income,Nil income
1,0.00708,0.110766
2,0.00708,0.067038
3,0.00708,0.047001
4,0.00708,0.043732
5,0.00708,0.040434
6,0.00708,0.051671
7,0.00708,0.052206
8+,0.00708,0.031808


## Train or reload the model

In [4]:
if TRAIN_MODEL:
    training_result = train_household_wgan(prepared, config=CONFIG)
    model = training_result.model
    metadata = {
        'attributes': list(HOUSEHOLD_ATTRIBUTES),
        'conditioning': ['hhsize', 'hhinc'],
        'negative_income_assumption': prepared.diagnostics['assumption'],
        'negative_income_probability': prepared.negative_income_probability,
        'seed_file': str(SEED_FILE.relative_to(REPO_ROOT)),
        'seed_sha256': file_sha256(SEED_FILE),
        'census_file': str(CENSUS_FILE.relative_to(REPO_ROOT)),
        'census_sha256': file_sha256(CENSUS_FILE),
        'diagnostics': training_result.diagnostics,
        'versions': {'torch': torch.__version__, 'pandas': pd.__version__, 'numpy': np.__version__},
    }
else:
    model, metadata = load_wgan_artifact(MODEL_FILE)
print('Device:', model.device)
print('Model trained' if TRAIN_MODEL else f'Loaded {MODEL_FILE}')

Device: mps
Model trained


## Training and pilot diagnostics

In [5]:
losses = pd.DataFrame({'generator': model.g_losses, 'critic': model.d_losses})
display(losses.tail())
assert np.isfinite(losses.to_numpy()).all(), 'Training produced a non-finite loss'
pilot_targets = {state: 2_000 for state in model.state_names['hhsize']}
from PopSynthesis.Methods.GAN.wgan import sample_exact_marginal
pilot = sample_exact_marginal(model, 'hhsize', pilot_targets, seed=RANDOM_SEED, chunk_size=CHUNK_SIZE)
display(pilot['hhinc'].value_counts(normalize=True).rename('probability').to_frame())
diversity = pilot[list(GENERATED_ATTRIBUTES)].nunique().rename('unique_states')
display(diversity.to_frame())
assert 'Negative income' in set(pilot['hhinc'])
assert diversity.gt(1).all(), f'Collapsed generated attributes: {diversity[diversity <= 1].to_dict()}'
if TRAIN_MODEL:
    save_wgan_artifact(model, metadata, MODEL_FILE)
    print('Validated and wrote:', MODEL_FILE)

,generator,critic
95,-2.170045,-0.356022
96,-2.195647,-0.347070
97,-2.138307,-0.345529
98,-2.191607,-0.346615
99,-2.176521,-0.345568


,probability
hhinc,
2500-2999,0.109000
2000-2499,0.094875
1750-1999,0.071375
1250-1499,0.069125
1500-1749,0.066875
1000-1249,0.065500
3500-3999,0.058750
800-999,0.057938
Nil income,0.055375


,unique_states
dwelltype,5
owndwell,4
totalvehs,5


Validated and wrote: /Users/bobla/Documents/repos/PopSyn_Monash/IO/models/hh_wgan/household_wgan.pt


## Generate the census-household-size-constrained population

In [6]:
zone_column = next(column for column in census.columns if column[0] == 'zone_id')
hhsize_columns = [column for column in census.columns if column[0] == 'hhsize']
targets = census.set_index(zone_column)[hhsize_columns].copy()
targets.columns = [str(column[1]) for column in hhsize_columns]
targets.index.name = 'zone_id'
population = sample_zonal_exact_marginal(
    model=model, attribute='hhsize', zone_targets=targets,
    zone_field='zone_id', seed=RANDOM_SEED, chunk_size=CHUNK_SIZE,
)
actual = population.groupby(['zone_id', 'hhsize']).size().unstack(fill_value=0).reindex(
    index=targets.index, columns=targets.columns, fill_value=0
).astype('int64')
actual.index.name, actual.columns.name = targets.index.name, targets.columns.name
pd.testing.assert_frame_equal(actual, targets.astype('int64'))
assert 'Negative income' in set(population['hhinc'])
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
population.to_csv(OUTPUT_FILE, index=False)
print(f'Wrote {len(population):,} households to {OUTPUT_FILE}')
display(population.head())

Wrote 2,420,221 households to /Users/bobla/Documents/repos/PopSyn_Monash/IO/output/hh_wgan_exact_hhsize.csv


,zone_id,dwelltype,owndwell,hhinc,totalvehs,hhsize
0,3000,Separate House,Being Rented,500-649,1,1
1,3000,Flat or Apartment,Being Purchased,800-999,1,1
2,3000,Separate House,Being Purchased,500-649,1,1
3,3000,Other,Fully Owned,500-649,1,1
4,3000,Separate House,Fully Owned,1250-1499,1,1
